# Full BRISC pipeline smoke test

`manifest → split → train ResNet18 + U-Net → save checkpoints → validation/test inference → validation calibration → frozen-threshold test fusion`


In [13]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data import CLASS_NAMES
from prepare_plane_split import build_aligned_manifest, build_experiment_split

SEED = 42
SPLIT_ROOT = PROJECT_ROOT / "data" / "experiments" / "pipeline_smoke_seed42"
TRAIN_OUTPUT_ROOT = PROJECT_ROOT / "results" / "pipeline_smoke" / "training"
PREDICTION_ROOT = PROJECT_ROOT / "results" / "pipeline_smoke" / "predictions"
CHECKPOINT_ROOT = PROJECT_ROOT / "checkpoints" / "pipeline_smoke"
REPORT_ROOT = PROJECT_ROOT / "results" / "pipeline_smoke" / "reports"
for directory in (SPLIT_ROOT, PREDICTION_ROOT, REPORT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src")
environment["NO_ALBUMENTATIONS_UPDATE"] = "1"
environment["OMP_NUM_THREADS"] = "1"

def run_step(name: str, arguments: list[str]) -> None:
    print(f"\n--- {name} ---")
    completed = subprocess.run(
        [sys.executable, *arguments],
        cwd=PROJECT_ROOT,
        env=environment,
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()

print(f"Python: {sys.executable}")
print(f"Project: {PROJECT_ROOT}")

Python: /opt/anaconda3/envs/brainTumor/bin/python
Project: /Users/louinsleung/Documents/brainTumor


## 1.a small real-data split

In [14]:
manifest = pd.read_csv(PROJECT_ROOT / "brisc2025" / "manifest.csv")
aligned = build_aligned_manifest(manifest)
full_split = build_experiment_split(
    aligned,
    held_out_plane="none",
    seed=SEED,
    validation_fraction=0.2,
)

def cap_each_stratum(frame: pd.DataFrame, size: int, seed: int) -> pd.DataFrame:
    parts = []
    for _, group in frame.groupby(["tumor_label", "plane_label"]):
        parts.append(group.sample(n=min(size, len(group)), random_state=seed))
    return pd.concat(parts, ignore_index=True)

caps = {"train": 12, "validation": 6, "test":6 }
smoke_parts = []
for split, cap in caps.items():
    partition = full_split.loc[full_split["experiment_split"] == split]
    smoke_parts.append(cap_each_stratum(partition, cap, SEED))
smoke_split = pd.concat(smoke_parts, ignore_index=True)
split_file = SPLIT_ROOT / "split_manifest.csv"
smoke_split.to_csv(split_file, index=False)

counts = smoke_split.groupby(
    ["experiment_split", "tumor_label", "plane_label"]
).size().rename("images")
assert set(smoke_split["experiment_split"]) == {"train", "validation", "test"}
assert smoke_split.loc[
    smoke_split["tumor_label"] != "no_tumor", "mask_relative_path"
].notna().all()
print(smoke_split["experiment_split"].value_counts())
counts.to_frame()

experiment_split
train         144
validation     72
test           72
Name: count, dtype: int64


images
experiment_split tumor_label plane_label        
test             glioma      axial             6
                             coronal           6
                             sagittal          6
                 meningioma  axial             6
                             coronal           6
                             sagittal          6
                 no_tumor    axial             6
                             coronal           6
                             sagittal          6
                 pituitary   axial             6
                             coronal           6
                             sagittal          6
train            glioma      axial            12
                             coronal          12
                             sagittal         12
                 meningioma  axial            12
                             coronal          12
                             sagittal         12
                 no_tumor    axial            12
                             coronal          12
                             sagittal         12
                 pituitary   axial            12
                             coronal          12
                             sagittal         12
validation       glioma      axial             6
                             coronal           6
                             sagittal          6
                 meningioma  axial             6
                             coronal           6
                             sagittal          6
                 no_tumor    axial             6
                             coronal           6
                             sagittal          6
                 pituitary   axial             6
                             coronal           6
                             sagittal          6

## 2. Train models briefly

In [15]:
run_step(
    "Train ResNet18 classifier",
    [
        "src/train_classification.py",
        "--data-root", str(PROJECT_ROOT / "brisc2025"),
        "--split-root", str(SPLIT_ROOT),
        "--output-dir", str(TRAIN_OUTPUT_ROOT),
        "--checkpoint-dir", str(CHECKPOINT_ROOT),
        "--head-epochs", "2",
        "--fine-tune-epochs", "20",
        "--earlyStopping", "5",
        "--run-name", "classification",
        "--image-size", "256",
        "--batch-size", "8",
        "--head-epochs", "2",
        "--earlyStopping",
        "--num-workers", "0",
        "--device", "cpu",
        "--seed", str(SEED),
    ],
)


--- Train ResNet18 classifier ---
Run: classification
Device: cpu; train=144, validation=72
Epoch 01 [head]: train loss=1.3341, F1=0.3519; val loss=1.2658, F1=0.1286, accuracy=0.2639
Epoch 02 [head]: train loss=1.1143, F1=0.5187; val loss=1.0321, F1=0.5958, accuracy=0.6111
Epoch 03 [fine_tune]: train loss=0.9740, F1=0.5742; val loss=0.5621, F1=0.7939, accuracy=0.7917
Epoch 04 [fine_tune]: train loss=0.3483, F1=0.8906; val loss=0.5950, F1=0.7834, accuracy=0.7778
Epoch 05 [fine_tune]: train loss=0.2309, F1=0.9174; val loss=0.4803, F1=0.8087, accuracy=0.8056
Epoch 06 [fine_tune]: train loss=0.2779, F1=0.9170; val loss=0.4299, F1=0.8043, accuracy=0.8056
Epoch 07 [fine_tune]: train loss=0.1052, F1=0.9651; val loss=0.4398, F1=0.8229, accuracy=0.8194
Epoch 08 [fine_tune]: train loss=0.1192, F1=0.9512; val loss=0.6944, F1=0.7552, accuracy=0.7639
Epoch 09 [fine_tune]: train loss=0.0621, F1=0.9931; val loss=0.4743, F1=0.8198, accuracy=0.8194
Epoch 10 [fine_tune]: train loss=0.1456, F1=0.9445; v

In [16]:
run_step(
    "Train U-Net segmenter",
    [
        "src/train_segmentation.py",
        "--data-root", str(PROJECT_ROOT / "brisc2025"),
        "--split-root", str(SPLIT_ROOT),
        "--output-dir", str(TRAIN_OUTPUT_ROOT),
        "--checkpoint-dir", str(CHECKPOINT_ROOT),
        "--run-name", "segmentation",
        "--image-size", "256",
        "--batch-size", "4",
        "--epochs", "20",
        "--earlyStopping", "5",
        "--num-workers", "0",
        "--device", "cpu",
        "--seed", str(SEED),
    ],
)

classification_checkpoint = CHECKPOINT_ROOT / "classification" / "best.pt"
segmentation_checkpoint = CHECKPOINT_ROOT / "segmentation" / "best.pt"
assert classification_checkpoint.exists()
assert segmentation_checkpoint.exists()
classification_checkpoint, segmentation_checkpoint


--- Train U-Net segmenter ---
Run: segmentation
Device: cpu; train=144, validation=72
Epoch 01: train loss=0.6753, positive Dice=0.0261; val loss=0.6090, positive Dice=0.0000, all-case Dice=0.2500
Epoch 02: train loss=0.5848, positive Dice=0.0140; val loss=0.5755, positive Dice=0.0000, all-case Dice=0.2500
Epoch 03: train loss=0.5550, positive Dice=0.0105; val loss=0.5366, positive Dice=0.0000, all-case Dice=0.2500
Epoch 04: train loss=0.5340, positive Dice=0.0023; val loss=0.5265, positive Dice=0.0000, all-case Dice=0.2500
Epoch 05: train loss=0.5258, positive Dice=0.0015; val loss=0.5406, positive Dice=0.0539, all-case Dice=0.0543
Epoch 06: train loss=0.5111, positive Dice=0.0971; val loss=0.5091, positive Dice=0.1890, all-case Dice=0.1557
Epoch 07: train loss=0.4952, positive Dice=0.1924; val loss=0.5342, positive Dice=0.2270, all-case Dice=0.1703
Epoch 08: train loss=0.4748, positive Dice=0.2615; val loss=0.5969, positive Dice=0.1618, all-case Dice=0.1214
Epoch 09: train loss=0.46

(PosixPath('/Users/louinsleung/Documents/brainTumor/checkpoints/pipeline_smoke/classification/best.pt'),
 PosixPath('/Users/louinsleung/Documents/brainTumor/checkpoints/pipeline_smoke/segmentation/best.pt'))

## 3. predict validation and test splits

In [17]:
prediction_runs = {}
for task, checkpoint, batch_size in (
    ("classification", classification_checkpoint, 8),
    ("segmentation", segmentation_checkpoint, 4),
):
    for split in ("validation", "test"):
        output_dir = PREDICTION_ROOT / f"{task}_{split}"
        run_step(
            f"Predict {task} {split}",
            [
                "src/predict.py",
                "--task", task,
                "--checkpoint", str(checkpoint),
                "--split", split,
                "--split-root", str(SPLIT_ROOT),
                "--data-root", str(PROJECT_ROOT / "brisc2025"),
                "--output-dir", str(output_dir),
                "--batch-size", str(batch_size),
                "--device", "cpu",
            ],
        )
        prediction_runs[(task, split)] = output_dir / "run.json"

assert all(path.exists() for path in prediction_runs.values())
prediction_runs


--- Predict classification validation ---
Saved 72 validation predictions to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/classification_validation/run.json


--- Predict classification test ---
Saved 72 test predictions to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/classification_test/run.json


--- Predict segmentation validation ---
Saved 72 validation predictions to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/segmentation_validation/run.json


--- Predict segmentation test ---
Saved 72 test predictions to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/segmentation_test/run.json



{('classification',
  'validation'): PosixPath('/Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/classification_validation/run.json'),
 ('classification',
  'test'): PosixPath('/Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/classification_test/run.json'),
 ('segmentation',
  'validation'): PosixPath('/Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/segmentation_validation/run.json'),
 ('segmentation',
  'test'): PosixPath('/Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/predictions/segmentation_test/run.json')}

## 4. Select thresholds using validation predictions only

In [18]:
threshold_file = REPORT_ROOT / "thresholds.json"
run_step(
    "Calibrate validation thresholds",
    [
        "src/calibrate_thresholds.py",
        "--classification-run", str(prediction_runs[("classification", "validation")]),
        "--segmentation-run", str(prediction_runs[("segmentation", "validation")]),
        "--output-file", str(threshold_file),
    ],
)
thresholds = json.loads(threshold_file.read_text())
calibration_table = pd.DataFrame(
    [
        {
            "Model decision": "Classification tumor probability",
            "Selected threshold": thresholds["classification_threshold"],
            "Validation balanced accuracy": thresholds["classification_validation_balanced_accuracy"],
        },
        {
            "Model decision": "Segmentation mask area",
            "Selected threshold": thresholds["area_threshold"],
            "Validation balanced accuracy": thresholds["segmentation_validation_balanced_accuracy"],
        },
    ]
)
print("\nVALIDATION CALIBRATION")
print(calibration_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print(f"Validation samples: {thresholds['validation_sample_count']}")


--- Calibrate validation thresholds ---
Selected classification threshold 0.632205 and area threshold 0.00288391; saved /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/reports/thresholds.json


VALIDATION CALIBRATION
                  Model decision  Selected threshold  Validation balanced accuracy
Classification tumor probability              0.6322                        0.9722
          Segmentation mask area              0.0029                        0.7222
Validation samples: 72


## 5. Freeze thresholds, fuse test predictions, and compute final metrics

In [19]:
fusion_file = REPORT_ROOT / "test_fusion.json"
run_step(
    "Fuse test predictions",
    [
        "src/decision_fusion.py",
        "--classification-run", str(prediction_runs[("classification", "test")]),
        "--segmentation-run", str(prediction_runs[("segmentation", "test")]),
        "--classification-threshold", str(thresholds["classification_threshold"]),
        "--area-threshold", str(thresholds["area_threshold"]),
        "--output-file", str(fusion_file),
    ],
)
fusion_report = json.loads(fusion_file.read_text())
summary = fusion_report["summary"]
assert summary["sample_count"] == len(
    smoke_split.loc[smoke_split["experiment_split"] == "test"]
)
assert summary["classification_threshold"] == thresholds["classification_threshold"]
assert summary["area_threshold"] == thresholds["area_threshold"]

from sklearn.metrics import accuracy_score, f1_score

classification_test = pd.DataFrame(
    json.loads(prediction_runs[("classification", "test")].read_text())["predictions"]
)
segmentation_test = pd.DataFrame(
    json.loads(prediction_runs[("segmentation", "test")].read_text())["predictions"]
)
fusion_samples = pd.DataFrame(fusion_report["samples"])

def classification_metrics(frame: pd.DataFrame) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(frame["tumor_label"], frame["predicted_class"])),
        "macro_f1": float(
            f1_score(
                frame["tumor_label"],
                frame["predicted_class"],
                labels=list(CLASS_NAMES),
                average="macro",
                zero_division=0,
            )
        ),
    }

def segmentation_metrics(frame: pd.DataFrame) -> dict[str, float]:
    positive = frame.loc[frame["tumor_label"] != "no_tumor"]
    return {
        "mean_dice_all": float(frame["mask_dice"].mean()),
        "mean_iou_all": float(frame["mask_iou"].mean()),
        "mean_dice_positive": float(positive["mask_dice"].mean()),
        "mean_iou_positive": float(positive["mask_iou"].mean()),
    }

covered_ids = set(
    fusion_samples.loc[~fusion_samples["is_uncertain"], "filename"]
)
covered_classification = classification_test.loc[
    classification_test["filename"].isin(covered_ids)
]
covered_segmentation = segmentation_test.loc[
    segmentation_test["filename"].isin(covered_ids)
]
evaluation = {
    "sample_count": len(classification_test),
    "classification": classification_metrics(classification_test),
    "segmentation": segmentation_metrics(segmentation_test),
    "fusion": {
        "coverage": float(len(covered_ids) / len(classification_test)),
        "agreement_rate": summary["agreement_rate"],
        "uncertain_rate": summary["uncertain_rate"],
        "covered_classification": classification_metrics(covered_classification),
        "covered_segmentation": segmentation_metrics(covered_segmentation),
    },
    "by_plane": {},
}
for plane in sorted(classification_test["plane_label"].unique()):
    evaluation["by_plane"][plane] = {
        "classification": classification_metrics(
            classification_test.loc[classification_test["plane_label"] == plane]
        ),
        "segmentation": segmentation_metrics(
            segmentation_test.loc[segmentation_test["plane_label"] == plane]
        ),
    }

evaluation_file = REPORT_ROOT / "test_metrics.json"
evaluation_file.write_text(json.dumps(evaluation, indent=2))
assert set(evaluation["by_plane"]) == {"axial", "coronal", "sagittal"}

overall_table = pd.DataFrame(
    [
        ("Test samples", evaluation["sample_count"]),
        ("Classification accuracy", evaluation["classification"]["accuracy"]),
        ("Classification macro F1", evaluation["classification"]["macro_f1"]),
        ("Segmentation Dice — all", evaluation["segmentation"]["mean_dice_all"]),
        ("Segmentation IoU — all", evaluation["segmentation"]["mean_iou_all"]),
        ("Segmentation Dice — tumor positive", evaluation["segmentation"]["mean_dice_positive"]),
        ("Segmentation IoU — tumor positive", evaluation["segmentation"]["mean_iou_positive"]),
    ],
    columns=["Metric", "Value"],
)
fusion_table = pd.DataFrame(
    [
        ("Coverage / agreement rate", evaluation["fusion"]["coverage"]),
        ("Uncertain rate", evaluation["fusion"]["uncertain_rate"]),
        ("Covered classification accuracy", evaluation["fusion"]["covered_classification"]["accuracy"]),
        ("Covered classification macro F1", evaluation["fusion"]["covered_classification"]["macro_f1"]),
        ("Covered positive-mask Dice", evaluation["fusion"]["covered_segmentation"]["mean_dice_positive"]),
        ("Covered positive-mask IoU", evaluation["fusion"]["covered_segmentation"]["mean_iou_positive"]),
    ],
    columns=["Metric", "Value"],
)
plane_rows = []
for plane, metrics in evaluation["by_plane"].items():
    plane_rows.append(
        {
            "Plane": plane.title(),
            "Classification accuracy": metrics["classification"]["accuracy"],
            "Classification macro F1": metrics["classification"]["macro_f1"],
            "Positive-mask Dice": metrics["segmentation"]["mean_dice_positive"],
            "Positive-mask IoU": metrics["segmentation"]["mean_iou_positive"],
        }
    )
plane_table = pd.DataFrame(plane_rows)

print("\nOVERALL TEST METRICS")
print(overall_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nFUSION AND COVERED-CASE METRICS")
print(fusion_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nMETRICS BY ANATOMICAL PLANE")
print(plane_table.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print(f"\nDetailed metrics saved to: {evaluation_file}")


--- Fuse test predictions ---
Saved 72 decisions and 18 inconsistencies to /Users/louinsleung/Documents/brainTumor/results/pipeline_smoke/reports/test_fusion.json


OVERALL TEST METRICS
                            Metric   Value
                      Test samples 72.0000
           Classification accuracy  0.8611
           Classification macro F1  0.8527
           Segmentation Dice — all  0.4347
            Segmentation IoU — all  0.3474
Segmentation Dice — tumor positive  0.4685
 Segmentation IoU — tumor positive  0.3521

FUSION AND COVERED-CASE METRICS
                         Metric  Value
      Coverage / agreement rate 0.7500
                 Uncertain rate 0.2500
Covered classification accuracy 0.8889
Covered classification macro F1 0.8744
     Covered positive-mask Dice 0.4928
      Covered positive-mask IoU 0.3720

METRICS BY ANATOMICAL PLANE
   Plane  Classification accuracy  Classification macro F1  Positive-mask Dice  Positive-mask IoU
   Axial                   0.7917   